# Data Analysis

Purpose: summarize the processed dataset, generate descriptive statistics, produce publication-quality figures and tables, and prepare inputs for Bayesian modeling.

In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.circular_statistics import (
    summarize_error,
    summarize_circular_error,
    summarize_error_by_setsize,
    summarize_error_by_participant,
    summarize_error_by_experiment,
)

from src.visualization import (
    save_figure,
    plot_setsize_distribution,
    plot_error_distribution,
    plot_error_density,
    plot_error_distribution_by_setsize,
    plot_circular_error,
    plot_error_by_setsize,
    plot_participant_precision,
    plot_experiment_setsize_error,
)

In [2]:
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "vandenberg12-clean.csv"

TABLES_DIR = PROJECT_ROOT / "results" / "tables"
FIGURES_DIR = PROJECT_ROOT / "results" / "figures"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
# Load Dataset

df = pd.read_csv(DATA_PATH)

print(df.head())

  experiment  id  trial   stims response_selection  setsize  target  response  \
0       Exp1   1      1  colors          scrolling        3     300       296   
1       Exp1   1      2  colors          scrolling        4     278       274   
2       Exp1   1      3  colors          scrolling        2     168       150   
3       Exp1   1      4  colors          scrolling        1     326       334   
4       Exp1   1      5  colors          scrolling        3     294       302   

   targetrad  responserad    devrad  errorrad  
0   5.235988     5.166175 -0.069813  0.069813  
1   4.852015     4.782202 -0.069813  0.069813  
2   2.932153     2.617994 -0.314159  0.314159  
3   5.689773     5.829400  0.139626  0.139626  
4   5.131268     5.270894  0.139626  0.139626  


In [4]:
# Data Validation

validation = pd.DataFrame(
    {
        "Missing values": [df.isna().sum().sum()],
        "Duplicated rows": [df.duplicated().sum()],
        "Minimum error": [df["errorrad"].min()],
        "Maximum error": [df["errorrad"].max()],
        "Expected maximum": [np.pi],
    }
)

print(validation)

validation.to_csv(
    TABLES_DIR / "data-validation.csv",
    index=False,
)

   Missing values  Duplicated rows  Minimum error  Maximum error  \
0               0                0            0.0       3.141593   

   Expected maximum  
0          3.141593  


In [5]:
# Dataset Overview

dataset_summary = pd.DataFrame(
    {
        "rows": [len(df)],
        "participants": [df["id"].nunique()],
        "participant_experiment_units": [df.groupby(["experiment", "id"]).ngroups],
        "experiments": [df["experiment"].nunique()],
        "trials": [len(df)],
        "variables": [df.shape[1]],
    }
)

print(dataset_summary)

# Save Dataset Summary

dataset_summary.to_csv(
    TABLES_DIR / "dataset-summary.csv",
    index=False,
)

    rows  participants  participant_experiment_units  experiments  trials  \
0  37824            13                            32            3   37824   

   variables  
0         12  


In [6]:
# Participant Structure

participant_summary = (
    df.groupby(["experiment", "id"])
    .size()
    .rename("n_trials")
    .reset_index()
)

print(participant_summary.head())

# Save Participant Summary

participant_summary.to_csv(
    TABLES_DIR / "participant-summary.csv",
    index=False,
)

  experiment  id  n_trials
0       Exp1   1       864
1       Exp1   2       864
2       Exp1   3       864
3       Exp1   4       864
4       Exp1   5       864


In [7]:
# Trial Counts by Experiment and Set Size

trial_counts = (
    df.groupby(["experiment", "setsize"])
    .size()
    .reset_index(name="n_trials")
)

trial_counts.to_csv(
    TABLES_DIR / "trial-counts.csv",
    index=False,
)

print(trial_counts)

   experiment  setsize  n_trials
0        Exp1        1      1404
1        Exp1        2      1404
2        Exp1        3      1404
3        Exp1        4      1404
4        Exp1        5      1404
5        Exp1        6      1404
6        Exp1        7      1404
7        Exp1        8      1404
8        Exp2        1      1920
9        Exp2        2      1920
10       Exp2        3      1920
11       Exp2        4      1920
12       Exp2        5      1920
13       Exp2        6      1920
14       Exp2        7      1920
15       Exp2        8      1920
16      ExpS3        1      1404
17      ExpS3        2      1404
18      ExpS3        3      1404
19      ExpS3        4      1404
20      ExpS3        5      1404
21      ExpS3        6      1404
22      ExpS3        7      1404
23      ExpS3        8      1404


In [8]:
# Set Size Distribution

setsize_summary = (
    df["setsize"]
    .value_counts()
    .sort_index()
)

print(setsize_summary)

# Figure

fig = plot_setsize_distribution(setsize_summary)

save_figure(
    fig,
    FIGURES_DIR,
    "setsize-distribution",
)

setsize
1    4728
2    4728
3    4728
4    4728
5    4728
6    4728
7    4728
8    4728
Name: count, dtype: int64


In [9]:
# Circular Error Summary

error_summary = summarize_error(df)

print(error_summary)

# Save Table

error_summary.to_csv(
    TABLES_DIR / "error-summary.csv"
)

# Error Figures

fig = plot_error_distribution(df["errorrad"])

save_figure(
    fig,
    FIGURES_DIR,
    "error-distribution"
)

fig = plot_error_density(df["errorrad"])

save_figure(
    fig,
    FIGURES_DIR,
    "error-density"
)

fig = plot_error_distribution_by_setsize(df)

save_figure(
    fig,
    FIGURES_DIR,
    "error-distribution-by-setsize",
)

            count      mean       std  min       25%       50%       75%  \
errorrad  37824.0  0.635962  0.730503  0.0  0.139626  0.349066  0.802851   

               max  
errorrad  3.141593  


In [10]:
# Circular Statistics

circular_error_summary = summarize_circular_error(df)

print(circular_error_summary)

circular_error_summary.to_csv(
    TABLES_DIR / "circular-error-summary.csv",
    index=False,
)

# Figure

fig = plot_circular_error(df)

save_figure(
    fig,
    FIGURES_DIR,
    "circular-error",
)

   circular_mean  circular_variance  circular_std
0       0.320209           0.362869      0.474753


In [11]:
# Error by Set Size

setsize_error = summarize_error_by_setsize(df)

print(setsize_error)

# Save Table

setsize_error.to_csv(
    TABLES_DIR / "setsize-error-summary.csv",
    index=False,
)

# Figure

fig = plot_error_by_setsize(setsize_error.rename(columns={"mean": "mean_error"}))

save_figure(
    fig,
    FIGURES_DIR,
    "error-by-setsize",
)

   setsize  count      mean    median       std  min       max
0        1   4728  0.250455  0.174533  0.235442  0.0  3.036873
1        2   4728  0.337903  0.244346  0.349411  0.0  3.054326
2        3   4728  0.443261  0.279253  0.517693  0.0  3.141593
3        4   4728  0.585018  0.349066  0.666385  0.0  3.141593
4        5   4728  0.691875  0.418879  0.751093  0.0  3.141593
5        6   4728  0.835990  0.506145  0.827345  0.0  3.141593
6        7   4728  0.938103  0.593412  0.880612  0.0  3.141593
7        8   4728  1.005089  0.663225  0.898984  0.0  3.141593


In [12]:
# Participant Precision

participant_precision = summarize_error_by_participant(df)

print(participant_precision.head())

# Save Table

participant_precision.to_csv(
    TABLES_DIR / "participant-precision-summary.csv",
    index=False,
)

# Figure

participant_precision = (
    participant_precision
    .sort_values("mean_error")
    .reset_index(drop=True)
)

fig = plot_participant_precision(participant_precision)

save_figure(
    fig,
    FIGURES_DIR,
    "participant-precision",
)

  experiment  id  count  mean_error    median       std
0       Exp1   1    864    0.420414  0.244346  0.516922
1       Exp1   2    864    0.529901  0.314159  0.591514
2       Exp1   3    864    0.538507  0.314159  0.645172
3       Exp1   4    864    0.677972  0.436332  0.699216
4       Exp1   5    864    0.672396  0.418879  0.723061


In [13]:
# Participant by Set Size

participant_setsize = (
    df.groupby(["id", "setsize"])["errorrad"]
    .agg(
        mean_error="mean",
        sd="std",
        n="count",
    )
    .reset_index()
)

# Save Table

participant_setsize.to_csv(
    TABLES_DIR / "participant-setsize-summary.csv",
    index=False,
)

In [14]:
# Experiment Comparison

experiment_summary = summarize_error_by_experiment(df)

print(experiment_summary)

# Save Table

experiment_summary.to_csv(
    TABLES_DIR / "experiment-comparison.csv",
    index=False,
)

# Experiment Setsize Summary

experiment_setsize_summary = (
    df.groupby(
        ["experiment", "setsize"]
    )["errorrad"]
    .mean()
    .reset_index()
)

# Save Table

experiment_setsize_summary.to_csv(
    TABLES_DIR / "experiment-setsize-summary.csv",
    index=False,
)

# Experiment Setsize Error

experiment_setsize_error = (
    df.groupby(
        ["experiment","setsize"]
    )["errorrad"]
    .mean()
    .reset_index()
    .rename(
        columns={
            "errorrad":"mean_error"
        }
    )
)

# Figure

fig = plot_experiment_setsize_error(
    experiment_setsize_error
)

save_figure(
    fig,
    FIGURES_DIR,
    "experiment-setsize-error",
)

  experiment  count      mean    median       std  min       max
0       Exp1  11232  0.564463  0.349066  0.651829  0.0  3.141593
1       Exp2  15360  0.679142  0.383972  0.762828  0.0  3.141593
2      ExpS3  11232  0.648410  0.349066  0.754089  0.0  3.141593


In [15]:
# Response Method Comparison

print(pd.crosstab(df["experiment"],df["response_selection"],))

# Save Table

response_method = pd.crosstab(
    df["experiment"],
    df["response_selection"]
)

response_method.to_csv(TABLES_DIR / "response-method-summary.csv")

response_selection  colorwheel  rotation  scrolling
experiment                                         
Exp1                         0         0      11232
Exp2                         0     15360          0
ExpS3                    11232         0          0


In [16]:
# Experiment by Set Size

print(
    df.groupby(["experiment", "setsize"])["errorrad"]
    .mean()
    .unstack()
)

# Save Table

experiment_setsize_summary.to_csv(
    TABLES_DIR / "experiment-setsize-summary.csv",
    index=False
)

setsize            1         2         3         4         5         6  \
experiment                                                               
Exp1        0.286562  0.322115  0.367141  0.472084  0.610443  0.689703   
Exp2        0.236374  0.350029  0.492374  0.651571  0.752319  0.884000   
ExpS3       0.233606  0.337107  0.452219  0.606937  0.690648  0.916621   

setsize            7         8  
experiment                      
Exp1        0.809017  0.958638  
Exp2        1.004173  1.062296  
ExpS3       0.976837  0.973307  
